# Clustering jerárquico con huecos: sueño, salud y estilo de vida

Completa las celdas siguientes para aplicar:

- clustering jerárquico
- dendrograma
- análisis de clústeres
- comparación con K-Means
- Utiliza para la resolución https://scikit-learn.org/stable/ y https://docs.scipy.org/doc/scipy/reference/cluster.hierarchy.html

>El origen del Dataset es https://www.kaggle.com/

## Contexto del dataset

El dataset recoge hábitos de sueño, salud y estilo de vida de aproximadamente 400 personas. Cada fila representa un individuo. Las variables que usaremos son:

| Variable | Tipo | Rango / Valores | Qué mide |
|---|---|---|---|
| Age | Numérica | 27 – 59 | Edad en años |
| Sleep Duration | Numérica | 5.8 – 8.5 | Horas de sueño por noche |
| Quality of Sleep | Numérica | 1 – 10 | Calidad subjetiva del sueño |
| Physical Activity Level | Numérica | 30 – 90 | Minutos de actividad física diaria |
| Stress Level | Numérica | 1 – 10 | Nivel de estrés autopercibido |
| Heart Rate | Numérica | 65 – 86 | Pulsaciones en reposo (bpm) |
| Daily Steps | Numérica | 3.000 – 10.000 | Pasos diarios |

También existen variables categóricas como `Gender` y `Sleep Disorder` que no entran en el clustering pero pueden usarse para interpretar los grupos.

---

**Antes de ejecutar nada, analiza:**  
¿Qué variable crees que tendrá más peso a la hora de separar los grupos?  
¿Esperas que el género influya en los clústeres resultantes?

Anota tu hipótesis aquí y compárala con los resultados al final.

> *Mi hipótesis:*

- Yo creo que las variables que más peso tendrán, serán; Age y Sleep Duration. Ya que las horas de sueño son clave cuando se trata de un dataset de habitos de sueño y la edad, porque puede variar bastante la calidad del sueño dependiendo de la edad.
- Yo creo que si influye, ya que el sueño puede variar dependiendo del genero.

## 1. Importación de librerías

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

## 2. Carga del dataset

In [2]:
df = pd.read_csv("Sleep_health_and_lifestyle_dataset.csv")
df.head()

,Person ID,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Blood Pressure,Heart Rate,Daily Steps,Sleep Disorder
0,1,Male,27,Software Engineer,6.1,6,42,6,Overweight,126/83,77,4200,NaN
1,2,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN
2,3,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN
3,4,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea
4,5,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea


## 2b. Comprobación y tratamiento de valores nulos

Antes de continuar, verifica si el dataset tiene valores que deban tener un tratamiento específico.  
Esta decisión afecta a todo el análisis posterior.

In [3]:
# Nulos por columna
print(df.isnull().sum())

Person ID                    0
Gender                       0
Age                          0
Occupation                   0
Sleep Duration               0
Quality of Sleep             0
Physical Activity Level      0
Stress Level                 0
BMI Category                 0
Blood Pressure               0
Heart Rate                   0
Daily Steps                  0
Sleep Disorder             219
dtype: int64


In [5]:
# ¿Qué porcentaje representa cada uno?
print((df['Sleep Disorder'].value_counts() / len(df) * 100).round(2))

Sleep Disorder
Sleep Apnea    20.86
Insomnia       20.59
Name: count, dtype: float64


In [ ]:
# Decide una estrategia y completa:
# Opción A — eliminar filas con nulos
# Opción B — imputar con la media
# Opción C — imputar con la mediana}


# df = df.____________________(__________________)   # Opción A
df["Sleep Disorder"] = df["Sleep Disorder"].fillna(df["Sleep Disorder"].mean) # Opción B o C

# Comprobación final: debe dar 0 en todas las columnas
print("Nulos restantes:", df.isnull().sum().sum())
#Elige la opción C, ya que tenemos 400 personas y si eliminar 219 que ya son mas de la mitad, romperiamos bastante el dataset.


Nulos restantes: 0


## 3. Exploración inicial

In [7]:
print("Filas y columnas:", df.shape)
print(df.columns.tolist())

Filas y columnas: (374, 13)
['Person ID', 'Gender', 'Age', 'Occupation', 'Sleep Duration', 'Quality of Sleep', 'Physical Activity Level', 'Stress Level', 'BMI Category', 'Blood Pressure', 'Heart Rate', 'Daily Steps', 'Sleep Disorder']


In [8]:
df.head()

,Person ID,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Blood Pressure,Heart Rate,Daily Steps,Sleep Disorder
0,1,Male,27,Software Engineer,6.1,6,42,6,Overweight,126/83,77,4200,<bound method Series.mean of 0 Na...
1,2,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,<bound method Series.mean of 0 Na...
2,3,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,<bound method Series.mean of 0 Na...
3,4,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea
4,5,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea


## 4. Selección de variables

Usa estas variables numéricas:
- Age
- Sleep Duration
- Quality of Sleep
- Physical Activity Level
- Stress Level
- Heart Rate
- Daily Steps

In [10]:
columnas = [
    "Age",
    "Sleep Duration",
    "Quality of Sleep",
    "Physical Activity Level",
    "Stress Level",
    "Heart Rate",
    "Daily Steps"
]

X = df[columnas]
X.head()

,Age,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,Heart Rate,Daily Steps
0,27,6.1,6,42,6,77,4200
1,28,6.2,6,60,8,75,10000
2,28,6.2,6,60,8,75,10000
3,28,5.9,4,30,8,85,3000
4,28,5.9,4,30,8,85,3000


## 5. Escalado

In [11]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## 6. Matriz de enlace

In [ ]:
Z = linkage(________________, method="________")
Z[:5]

## 7. Dendrograma

In [ ]:
plt.figure(figsize=(____, ____))
dendrogram(__________)
plt.title("_______________________________")
plt.xlabel("_______________________________")
plt.ylabel("_______________________________")
plt.show()

## 8. Número de clústeres

A partir del dendrograma, crea 3 clústeres.

In [ ]:
clusters_h = fcluster(____, t=____, criterion="____________")
df["cluster_jerarquico"] = ______________________

## 9. Tamaño de cada clúster

In [ ]:
df["cluster_jerarquico"]._______________________

## 10. Perfil medio de cada clúster

In [ ]:
perfil_h = df.groupby("_____________________")[_____________________].mean().round(2)
perfil_h

## 11. Relación con género

In [ ]:
pd.crosstab(df["_____________________"], df["_____________________"], normalize="index").round(3)

## 12. K-Means

In [ ]:
kmeans = KMeans(n_clusters=____, random_state=____, n_init=____)
clusters_k = kmeans.____________________(____________________)

df["cluster_kmeans"] = ____________________

## 13. Perfil medio en K-Means

In [ ]:
perfil_k = df.groupby("_____________________")[_____________________].mean().round(2)
perfil_k

## 14. Comparación entre métodos: Matriz de Confusión

> **Nota importante:** los números de clúster asignados por cada algoritmo son arbitrarios. El clúster "1" del jerárquico puede contener los mismos individuos que el clúster "2" de K-Means aunque sean el mismo grupo en la práctica. Interpreta la matriz fijándote en la **concentración de valores**, no en que coincidan los números.

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Construye la matriz cruzando ambos métodos
matriz = confusion_matrix(df["___________________"], df["___________________"])

# Visualización
plt.figure(figsize=(6, 4))
sns.heatmap(
    matriz,
    annot=____,
    fmt=____,
    cmap="____________________",
    xticklabels=["K1", "K2", "K3"],
    yticklabels=["H1", "H2", "H3"]
)
plt.xlabel("____________________")
plt.ylabel("____________________")
plt.title("Jerárquico vs K-Means")
plt.show()

**Reflexiona:** ¿Qué significa que haya valores altos fuera de la diagonal?  
¿Coinciden bien los dos métodos o hay grupos que se mezclan?

> *Mi respuesta:*

## 15. Visualización

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["_____________________"], df["_____________________"], c=df["_____________________"])
plt.xlabel("_____________________")
plt.ylabel("_____________________")
plt.title("_____________________")
plt.show()

## 16. Cuestiones finales

Responde:

1. ¿Cuántos clústeres observas en el dendrograma?
2. ¿Cómo describirías cada grupo?
3. ¿Qué variables distinguen mejor unos grupos de otros?
4. ¿Coinciden mucho los grupos jerárquicos y los de K-Means?
5. ¿Crees que las personas se agrupan más por género o por hábitos?
6. ¿Se cumplió tu hipótesis inicial sobre qué variable tendría más peso?